# 07 — Embeddings: documento semántico por persona y búsqueda semántica

**Objetivo (DEC-014, ver `context/DECISION_LOG.md`):** construir, para cada persona, un
**documento semántico** — un texto narrativo coherente que integra su trayectoria (dentro y
fuera de ESPOL), formación, docencia, investigación, funciones adicionales/subrogaciones,
capacitación, idiomas y reconocimientos — y calcular UN embedding por persona a partir de
ESE documento (no promediando embeddings de fragmentos sueltos, como en la versión anterior
de este notebook).

Arquitectura:

```
datos → información integrada de la persona → DOCUMENTO SEMÁNTICO → embedding → índice vectorial → búsqueda semántica
```

Este notebook es independiente de `06_clustering.ipynb`: no reentrena ni modifica el
clustering existente, y sus salidas (`data/embeddings/`) no se concatenan con
`X_modelado.csv`. Las secciones 1-4 (inventario de fuentes de texto y corpus por fragmento)
se conservan como estaban — ese corpus por fragmento se sigue usando para mostrar evidencia
("por qué es relevante") en el dashboard, pero **ya no es el insumo del embedding**; el
embedding se calcula sobre el documento semántico construido en la sección 5.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().parents[1] if (Path.cwd().name == "07_embeddings") else Path.cwd()
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_MODELING = ROOT / "data" / "modeling"
DATA_TRAYECTORIAS = ROOT / "data" / "trayectorias"
DATA_EMBEDDINGS = ROOT / "data" / "embeddings"
DATA_EMBEDDINGS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "notebooks" / "01_preprocesamiento"))
sys.path.insert(0, str(ROOT / "notebooks" / "07_embeddings"))
import _preprocesamiento_comun as pc
import _embeddings_comun as ec

RANDOM_STATE = 42

personas = pd.read_csv(DATA_MODELING / "personas_modelado.csv")
POBLACION = set(personas["IDPERSONA"])
print("Población de modelado (referencia para cobertura):", len(POBLACION))
print("Salidas de embeddings en:", DATA_EMBEDDINGS)

## 1. Columnas textuales disponibles en las fuentes institucionales

`data/features/*.csv` (usado en `03`-`05`) ya no tiene texto libre: todo fue agregado a conteos
numéricos por persona. El texto libre original vive en `data/processed/*.csv` (una fila por
publicación, capacitación, proyecto, etc., no por persona). Se revisan esas tablas en busca de
columnas de texto con contenido descriptivo real (no códigos, no IDs, no metadata administrativa).


In [ ]:
# Inventario de columnas candidatas identificadas por inspección de data/processed/*.csv
# (columnas de texto libre presentes en cada tabla, excluyendo IDs, fechas, referencias a archivos,
#  URLs, nombres/apellidos de personas y campos administrativos de auditoría/escalafón)
inventario_fuentes = pd.DataFrame([
    {"TABLA": "publicaciones.csv", "COLUMNA": "TITULO", "DESCRIPCION": "Título de publicación académica"},
    {"TABLA": "publicaciones.csv", "COLUMNA": "NOMBREREVISTA", "DESCRIPCION": "Nombre de la revista/venue (no describe el tema, sino el medio)"},
    {"TABLA": "proyecto_grado.csv", "COLUMNA": "NOMBRETRABAJOTITULACION", "DESCRIPCION": "Título del trabajo de titulación dirigido"},
    {"TABLA": "proyecto_grado.csv", "COLUMNA": "NOMBREPROGRAMA", "DESCRIPCION": "Programa académico del trabajo de titulación"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre del proyecto de investigación"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "STRAREACAMPOAMPLIO / STRAREAFRASCATI / STRSUBAREAFRASCATI", "DESCRIPCION": "Área de conocimiento del proyecto (texto descriptivo)"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "STRCAMPOESPECIFICO", "DESCRIPCION": "Contiene códigos numéricos ('1.0','2.0'...), no texto real (problema de calidad de datos)"},
    {"TABLA": "proyectos_vinculacion_disponible.csv", "COLUMNA": "NOMBREPROYECTO / NOMBREPROGRAMA", "DESCRIPCION": "Nombre del proyecto/programa de vinculación con la comunidad"},
    {"TABLA": "ponentes_todos.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Título de la ponencia/evento"},
    {"TABLA": "ponentes_todos.csv", "COLUMNA": "AREAACITACIONDESCRIPCION / TIPODESCRIPCION", "DESCRIPCION": "Códigos abreviados ('DI','PE','OT'), baja cobertura, no es texto descriptivo"},
    {"TABLA": "capacitaciones_todas.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre de la capacitación tomada"},
    {"TABLA": "capacitaciones_todas.csv", "COLUMNA": "AREACAPACITACIONDESCRIPCION", "DESCRIPCION": "Códigos abreviados, baja cobertura"},
    {"TABLA": "certificados_todos.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre de la certificación obtenida"},
    {"TABLA": "mencion_honor.csv", "COLUMNA": "NOMBREMENCION", "DESCRIPCION": "Categoría de la mención de honor (p.ej. 'Segunda mención'), no describe un tema"},
    {"TABLA": "experiencia_externa.csv", "COLUMNA": "CARGO", "DESCRIPCION": "Cargo ocupado en experiencia laboral externa"},
    {"TABLA": "experiencia_externa.csv", "COLUMNA": "INSTITUCION", "DESCRIPCION": "Nombre de la institución/empresa externa (identifica un organismo, no una competencia)"},
    {"TABLA": "carga_academica_disponible.csv", "COLUMNA": "NOMMATERIA", "DESCRIPCION": "Nombre de la materia impartida como docente"},
])
inventario_fuentes


## 2. Decisión: qué columnas se usan y por qué

Para cada tabla se mide, sobre la población de modelado (2213 personas), cuántas personas quedan
cubiertas por cada columna candidata, y se decide su inclusión con base en tres criterios: (a) si
describe realmente un **tema/competencia profesional o académica** (no un código, un medio o una
organización), (b) si tiene cobertura razonable, y (c) si no está prohibida por la consigna del
proyecto (IDs, nombres/apellidos, URLs, referencias a archivos, metadata administrativa).


In [ ]:
def cobertura(path, id_col, text_col, rename_id=None):
    df = pd.read_csv(DATA_PROCESSED / path, encoding="utf-8-sig")
    if rename_id:
        df = df.rename(columns={rename_id: id_col})
    df = df[df[id_col].isin(POBLACION) & df[text_col].notna()]
    return df[id_col].nunique(), len(df)

filas_decision = []

def registrar(fuente, tabla, columna, id_col, text_col, incluida, motivo, rename_id=None):
    n_personas, n_registros = cobertura(tabla, id_col, text_col, rename_id=rename_id)
    filas_decision.append({
        "FUENTE": fuente, "TABLA": tabla, "COLUMNA": text_col,
        "N_REGISTROS": n_registros, "N_PERSONAS_COBERTURA": n_personas,
        "PCT_POBLACION": round(100 * n_personas / len(POBLACION), 1),
        "INCLUIDA": incluida, "MOTIVO": motivo,
    })

registrar("PUBLICACION", "publicaciones.csv", "TITULO", "IDPERSONA", "TITULO",
          True, "Título describe directamente el tema de investigación")
registrar("PROYECTO_GRADO_DIRIGIDO", "proyecto_grado.csv", "NOMBRETRABAJOTITULACION", "IDPERSONA", "NOMBRETRABAJOTITULACION",
          True, "Título de tesis dirigida describe área de especialidad", rename_id="IDDIRECTOR")
registrar("PROYECTO_INVESTIGACION", "proyectos_investigacion_disponible.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de proyecto describe el tema investigado")
registrar("PROYECTO_VINCULACION", "proyectos_vinculacion_disponible.csv", "NOMBREPROYECTO", "IDPERSONA", "NOMBREPROYECTO",
          True, "Nombre de proyecto de vinculación describe el tema/comunidad de trabajo")
registrar("PONENCIA", "ponentes_todos.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Título de ponencia describe el tema presentado")
registrar("CAPACITACION", "capacitaciones_todas.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de capacitación describe el área de formación continua; cobertura muy alta")
registrar("CERTIFICACION", "certificados_todos.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de certificación describe una competencia adquirida")
registrar("EXPERIENCIA_EXTERNA_CARGO", "experiencia_externa.csv", "CARGO", "IDPERSONA", "CARGO",
          True, "Cargo externo describe rol/dominio profesional fuera de ESPOL")
registrar("MATERIA_IMPARTIDA", "carga_academica_disponible.csv", "NOMMATERIA", "IDPERSONA", "NOMMATERIA",
          True, "Nombre de materia impartida describe el área de docencia")
registrar("MENCION_HONOR", "mencion_honor.csv", "NOMBREMENCION", "IDPERSONA", "NOMBREMENCION",
          False, "Es una categoría de reconocimiento ('Diploma de honor'), no un tema/competencia")

decision_fuentes = pd.DataFrame(filas_decision).sort_values("PCT_POBLACION", ascending=False)
decision_fuentes


**Columnas descartadas explícitamente** (no llegan a la tabla de decisión porque no califican
como texto descriptivo, o violan las restricciones del alcance):

- `NOMBREREVISTA` (publicaciones) y `INSTITUCION` (experiencia externa): nombran un medio/organismo,
  no una competencia o tema — quedarían más cerca de "metadata administrativa" que de contenido
  profesional/académico.
- `STRCAMPOESPECIFICO` (proyectos de investigación): pese al nombre de la columna, contiene códigos
  numéricos como texto ('1.0', '2.0'), no descripciones — problema de calidad de datos, no una
  fuente de texto real.
- `AREAACITACIONDESCRIPCION`, `TIPODESCRIPCION` (ponentes) y `AREACAPACITACIONDESCRIPCION`
  (capacitaciones): son códigos abreviados ('DI', 'PE', 'OT') con cobertura muy baja, no texto
  descriptivo utilizable.
- Todo identificador (`IDPERSONA`, `IDCAPACITACION`, etc.), nombre/apellido de persona
  (`NOMBRES`, `APELLIDOS` en `heteroevaluacion_disponible.csv`), URL (`URLPUBLICACION`), referencia
  a archivo (`REFARCHIVO*`, `NAMEARCHDOC`) y campo administrativo de auditoría/escalafón
  (`ENESCALAFON`, `REVISADOPARAESCALAFON`, `IDUSUARIO`, `INGRESORRHH`, `ORIGENINGRESO`,
  `FECHASUBIDAARCHIVO`, etc.) — excluidos por instrucción explícita del proyecto.
- No se encontraron campos de comentarios/evaluaciones cualitativas de CENACAD en las tablas
  disponibles en `data/processed/` (solo el promedio numérico de heteroevaluación, ya incorporado
  en `X_modelado` como `PROMEDIO_HETEROEVALUACION`); si existen en otra fuente institucional no
  integrada aún, deben añadirse como una fuente adicional en una futura iteración.


## 3. Construcción del corpus por fragmento (evidencia/UI, no el embedding)

Se construye una tabla de detalle (una fila por *registro* de texto: cada capacitación,
publicación, proyecto, etc. es una fila separada). **Desde DEC-014, esta tabla ya no
alimenta el embedding** (ver sección 5) — se conserva porque el dashboard la usa para
mostrar fragmentos de evidencia ("por qué esta persona es relevante para tu búsqueda") sin
tener que re-parsear el documento semántico completo.

In [ ]:
def cargar(path):
    return pd.read_csv(DATA_PROCESSED / path, encoding="utf-8-sig")

registros = []

def agregar(df, id_col, fuente, texto_fn, rename_id=None):
    if rename_id:
        df = df.rename(columns={rename_id: id_col})
    df = df[df[id_col].isin(POBLACION)]
    for _, r in df.iterrows():
        texto = texto_fn(r)
        if texto and pd.notna(texto) and str(texto).strip():
            registros.append((int(r[id_col]), fuente, str(texto).strip()))

agregar(cargar("publicaciones.csv"), "IDPERSONA", "PUBLICACION",
        lambda r: r["TITULO"] if pd.notna(r["TITULO"]) else None)

agregar(cargar("proyecto_grado.csv"), "IDPERSONA", "PROYECTO_GRADO_DIRIGIDO",
        lambda r: (r["NOMBRETRABAJOTITULACION"] + (f" ({r['NOMBREPROGRAMA']})" if pd.notna(r.get("NOMBREPROGRAMA")) else ""))
                  if pd.notna(r["NOMBRETRABAJOTITULACION"]) else None,
        rename_id="IDDIRECTOR")

agregar(cargar("proyectos_investigacion_disponible.csv"), "IDPERSONA", "PROYECTO_INVESTIGACION",
        lambda r: " — ".join([str(r["NOMBRE"])] + [str(r[c]) for c in
                  ["STRAREACAMPOAMPLIO", "STRAREAFRASCATI", "STRSUBAREAFRASCATI"] if pd.notna(r.get(c))])
                  if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("proyectos_vinculacion_disponible.csv"), "IDPERSONA", "PROYECTO_VINCULACION",
        lambda r: r["NOMBREPROYECTO"] + (f" — {r['NOMBREPROGRAMA']}" if pd.notna(r.get("NOMBREPROGRAMA")) else "")
                  if pd.notna(r["NOMBREPROYECTO"]) else None)

agregar(cargar("ponentes_todos.csv"), "IDPERSONA", "PONENCIA",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("capacitaciones_todas.csv"), "IDPERSONA", "CAPACITACION",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("certificados_todos.csv"), "IDPERSONA", "CERTIFICACION",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("experiencia_externa.csv"), "IDPERSONA", "EXPERIENCIA_EXTERNA_CARGO",
        lambda r: r["CARGO"] if pd.notna(r["CARGO"]) else None)

df_materias = cargar("carga_academica_disponible.csv")
df_materias = df_materias[df_materias["IDPERSONA"].isin(POBLACION) & df_materias["NOMMATERIA"].notna()]
df_materias = df_materias.drop_duplicates(subset=["IDPERSONA", "NOMMATERIA"])
agregar(df_materias, "IDPERSONA", "MATERIA_IMPARTIDA", lambda r: r["NOMMATERIA"])

corpus_detalle = pd.DataFrame(registros, columns=["IDPERSONA", "FUENTE", "TEXTO"])
corpus_detalle = corpus_detalle.drop_duplicates(subset=["IDPERSONA", "FUENTE", "TEXTO"]).reset_index(drop=True)

print("Registros de texto totales:", len(corpus_detalle))
print("Personas con al menos un registro de texto:", corpus_detalle["IDPERSONA"].nunique(),
      f"de {len(POBLACION)} ({100*corpus_detalle['IDPERSONA'].nunique()/len(POBLACION):.1f}%)")
corpus_detalle["FUENTE"].value_counts()


In [ ]:
corpus_detalle.to_csv(DATA_EMBEDDINGS / "corpus_texto_detalle.csv", index=False)
print("Guardado:", DATA_EMBEDDINGS / "corpus_texto_detalle.csv", "-", corpus_detalle.shape)


## 4. Diagnóstico de cobertura del corpus

Se resume, por persona, cuántos registros de texto tiene y de cuántas fuentes distintas, **sin**
guardar el texto en este resumen (el texto vive únicamente en `corpus_texto_detalle.csv`, que es
insumo de trabajo, no un producto final).


In [ ]:
resumen_personas = (
    corpus_detalle.groupby("IDPERSONA")
    .agg(N_REGISTROS_TEXTO=("TEXTO", "size"), N_FUENTES_DISTINTAS=("FUENTE", "nunique"))
    .reset_index()
)
resumen_personas["LONGITUD_TOTAL_CARACTERES"] = (
    corpus_detalle.groupby("IDPERSONA")["TEXTO"].apply(lambda s: s.str.len().sum()).values
)

cobertura_completa = personas[["IDPERSONA"]].merge(resumen_personas, on="IDPERSONA", how="left")
cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]] = (
    cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]].fillna(0)
)
cobertura_completa.to_csv(DATA_EMBEDDINGS / "cobertura_texto_personas.csv", index=False)

sin_texto = (cobertura_completa["N_REGISTROS_TEXTO"] == 0).sum()
print(f"Personas sin ningún registro de texto: {sin_texto} ({100*sin_texto/len(cobertura_completa):.1f}%)")
cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(cobertura_completa["N_REGISTROS_TEXTO"].clip(upper=100), bins=40, color="#4C72B0")
axes[0].set_title("Registros de texto por persona (recortado en 100)")
axes[0].set_xlabel("N° de registros de texto")

fuente_counts = corpus_detalle["FUENTE"].value_counts()
axes[1].barh(fuente_counts.index[::-1], fuente_counts.values[::-1], color="#55A868")
axes[1].set_title("Registros de texto por fuente")
axes[1].set_xlabel("N° de registros")

plt.tight_layout()
plt.show()


**Lectura del diagnóstico:** el corpus cubre ~99% de la población de modelado con al menos un
registro de texto, dominado en volumen por `CAPACITACION` (más de la mitad de los registros) seguido
de `PUBLICACION`, `PROYECTO_INVESTIGACION`, `EXPERIENCIA_EXTERNA_CARGO` y `PROYECTO_GRADO_DIRIGIDO`.
Esto confirma que **sí existe suficiente texto institucional relevante** como para justificar
explorar una representación semántica — pero también advierte que, al calcular los embeddings, no
conviene concatenar todo el texto de una persona en un solo string (algunas personas superan 10,000
palabras): la mayoría de los modelos de embeddings truncan a unos pocos cientos de tokens. El diseño
más apropiado es **calcular un embedding por registro de texto y luego agregarlo (p.ej. promedio)
por persona**, posiblemente ponderando por fuente para que `CAPACITACION` no domine desproporcionadamente
solo por su volumen. Esta decisión de diseño se aplicará en la siguiente iteración, junto con el
modelo de embeddings elegido.


## 5. Documento semántico por persona (DEC-014)

Antes de calcular ningún embedding, se construye el **documento semántico**: un texto por
persona, ensamblado por secciones con información real (no se fuerzan secciones vacías),
que integra:

- **Trayectoria** — línea de tiempo cronológica combinando cargo estructural dentro de
  ESPOL (`tramos_rol.csv`, DEC-011), funciones adicionales/subrogaciones (`funciones_
  adicionales_persona.csv`, DEC-012, excluyendo las que coinciden con el contrato vigente)
  y experiencia laboral externa (`experiencia_externa.csv`) — en ese orden temporal real
  cuando las fechas lo permiten; no se inventan secuencias si faltan fechas.
- **Formación académica** (solo titulaciones con `Estado='Graduado'`).
- **Docencia** (materias impartidas, deduplicadas, con rango de años).
- **Investigación** (proyectos, publicaciones, tesis dirigidas, ponencias).
- **Vinculación** (proyectos de vinculación con la sociedad).
- **Capacitación** (cursos y certificaciones).
- **Idiomas** (excluyendo la lengua nativa).
- **Reconocimientos** (menciones de honor).

Implementado en `_embeddings_comun.py` (`construir_documentos_semanticos`), **separado**
de `_preprocesamiento_comun.py` para no mezclar esta lógica con las features del
clustering. Reutiliza `pc.construir_eventos_trayectoria` (ya calculada en
`04_trayectorias.ipynb`) en vez de reconstruir la línea de tiempo desde cero.

**Qué NO entra al documento (y por qué):**

| Se descarta de | Ejemplos | Motivo |
|---|---|---|
| El texto (queda solo en la fuente original) | IDs, `RMU`/salario, URLs, referencias a archivo, campos de auditoría (`ENESCALAFON`, `IDUSUARIO`...), nombres/apellidos | Sin valor semántico para búsqueda por tema/experiencia, o dato sensible/administrativo (ver DEC-003 sobre privacidad) |
| El texto (queda como metadata estructurada aparte) | `IDPERSONA`, `SECCIONES_INCLUIDAS`, `N_PALABRAS` | Útiles para filtrar/depurar resultados de búsqueda, pero no aportan significado semántico en sí mismos |
| El texto (se narra distinto) | Menciones de honor con categoría genérica sin tema | Se narran como reconocimiento con fecha/institución, no como "tema" de búsqueda |

In [ ]:
eventos_trayectoria = pd.read_csv(DATA_TRAYECTORIAS / "eventos_trayectoria_persona.csv")
# pd.to_datetime explicito, no parse_dates de read_csv (ver nota de robustez en
# construir_eventos_trayectoria / _preprocesamiento_comun.py, misma leccion de esta sesion).
for _c in ["FECHA_INICIO", "FECHA_FIN"]:
    eventos_trayectoria[_c] = pd.to_datetime(eventos_trayectoria[_c], format="mixed", errors="coerce")

# Feature engineering de movilidad de carrera (ver DECISION_LOG.md): ENTROPIA_CATEGORIA_CARGO
# habilita una oracion condicional de "alta diversidad de roles" solo para el tercil
# superior de la poblacion (ver _seccion_trayectoria) - se pasa aparte de eventos_trayectoria
# porque vive en features_trayectoria_persona.csv (04_trayectorias.ipynb), no en la tabla de
# eventos.
diversidad_trayectoria = pd.read_csv(
    DATA_TRAYECTORIAS / "features_trayectoria_persona.csv",
    usecols=["IDPERSONA", "ENTROPIA_CATEGORIA_CARGO", "N_CATEGORIAS_ROL_DISTINTAS",
             "TURBULENCIA_TRAMOS", "DURACION_MEDIANA_TRAMO_ANIOS"],
)

documentos = ec.construir_documentos_semanticos(POBLACION, eventos_trayectoria, diversidad_trayectoria)

documentos.to_csv(DATA_EMBEDDINGS / "documento_semantico_persona.csv", index=False, encoding="utf-8-sig")

vacios = (documentos["N_SECCIONES"] == 0).sum()
recortados = (documentos["SECCIONES_RECORTADAS"] != "").sum()
print(f"Guardado: {DATA_EMBEDDINGS / 'documento_semantico_persona.csv'} — {documentos.shape}")
print(f"Personas sin ninguna sección con información (documento vacío): {vacios} ({100*vacios/len(documentos):.1f}%)")
print(f"Personas con alguna sección recortada por presupuesto de longitud: {recortados} ({100*recortados/len(documentos):.1f}%)")
print()
print("Distribución de secciones incluidas por persona:")
print(documentos["N_SECCIONES"].value_counts().sort_index())
print()
print("Palabras por documento:")
print(documentos["N_PALABRAS"].describe())
print()
print("--- Ejemplo de documento (persona con más secciones) ---")
ejemplo = documentos.loc[documentos["N_SECCIONES"].idxmax()]
print(f"IDPERSONA {ejemplo['IDPERSONA']} ({ejemplo['N_PALABRAS']} palabras, secciones: {ejemplo['SECCIONES_INCLUIDAS']})")
print(ejemplo["DOCUMENTO_TEXTO"])

## 6. Cálculo del embedding (un vector por persona, a partir del documento semántico)

**Cambio de modelo respecto a la versión anterior de este notebook (decisión técnica,
DEC-014):** el modelo previo, `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`,
tiene un límite de contexto de **128 tokens** — adecuado para los fragmentos cortos que se
promediaban antes, pero insuficiente para un documento narrativo completo (mediana ~170
palabras, algunos documentos superan 300-400 palabras incluso después de recortar por
presupuesto en la sección 5): el texto se truncaría casi de inmediato, perdiendo la mayor
parte del contenido. Se cambia a **`intfloat/multilingual-e5-base`** (768 dimensiones,
contexto de **512 tokens**, multilingüe, entrenado específicamente para tareas de
recuperación/búsqueda semántica — no solo similitud de parafraseo). Sigue siendo un modelo
local (`sentence-transformers`), sin API key, sin enviar datos a terceros — se mantiene el
principio de privacidad de DEC-002. Los modelos E5 requieren un prefijo distinto para
documentos ("passage: ") y para consultas de búsqueda ("query: "); se aplica de forma
consistente aquí y en `notebooks/08_dashboard/app.py`.

A diferencia de la versión anterior, **no hay agregación**: cada persona tiene un único
texto (su documento semántico) y por lo tanto un único embedding directo — no se promedian
fragmentos ni se pondera por fuente.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-base"
model = SentenceTransformer(MODEL_NAME)

con_texto = documentos[documentos["DOCUMENTO_TEXTO"].str.len() > 0].copy()
sin_embedding = sorted(POBLACION - set(con_texto["IDPERSONA"]))

textos_passage = ("passage: " + con_texto["DOCUMENTO_TEXTO"]).tolist()
print(f"Codificando {len(textos_passage)} documentos (uno por persona)...")

embeddings_matrix = model.encode(
    textos_passage, batch_size=32, show_progress_bar=True, normalize_embeddings=True,
)
DIM_EMBEDDING = embeddings_matrix.shape[1]
ids_con_texto = con_texto["IDPERSONA"].to_numpy()

print("Embeddings calculados:", embeddings_matrix.shape)
print(f"Personas SIN embedding (documento vacío, ninguna sección con información): {len(sin_embedding)}")

In [ ]:
embeddings_personas = pd.DataFrame(
    embeddings_matrix, columns=[f"E_{i:03d}" for i in range(DIM_EMBEDDING)],
)
embeddings_personas.insert(0, "IDPERSONA", ids_con_texto)

embeddings_personas.to_csv(DATA_EMBEDDINGS / "embeddings_personas.csv", index=False)

print(f"Guardado: {DATA_EMBEDDINGS / 'embeddings_personas.csv'} — {embeddings_personas.shape}")
print(f"Personas SIN embedding (documento vacío): {len(sin_embedding)}")
print("IDs:", sin_embedding[:10], "..." if len(sin_embedding) > 10 else "")

In [ ]:
metadata_embeddings = pd.DataFrame([{
    "MODELO": MODEL_NAME,
    "DIMENSIONES": DIM_EMBEDDING,
    "METODO": "un embedding por persona, calculado directamente sobre su documento semántico "
              "completo (ver sección 5) - sin agregación de fragmentos (DEC-014)",
    "PREFIJO_DOCUMENTO": "passage: ",
    "PREFIJO_CONSULTA_BUSQUEDA": "query: ",
    "N_PERSONAS_CON_EMBEDDING": len(ids_con_texto),
    "N_PERSONAS_SIN_EMBEDDING": len(sin_embedding),
    "PALABRAS_PROMEDIO_DOCUMENTO": round(documentos["N_PALABRAS"].mean(), 1),
}])
metadata_embeddings.to_csv(DATA_EMBEDDINGS / "embeddings_metadata.csv", index=False)
metadata_embeddings.T

### Validación cualitativa: vecinos más cercanos en el espacio semántico

Antes de cualquier análisis cuantitativo, una revisión cualitativa rápida: para una persona con
suficiente texto (varias publicaciones/proyectos), ¿sus vecinos más cercanos por similitud coseno
tienen un perfil temático parecido? Esto no es una prueba formal, es una verificación de sentido
común de que el embedding capturó algo razonable.


In [ ]:
doc_por_persona = documentos.set_index("IDPERSONA")["DOCUMENTO_TEXTO"]
candidatos = documentos[documentos["N_SECCIONES"] >= 5]["IDPERSONA"]
persona_ejemplo = int(pd.Series(sorted(set(candidatos) & set(ids_con_texto))).sample(1, random_state=7).iloc[0])

idx_ejemplo = list(ids_con_texto).index(persona_ejemplo)
sims = embeddings_matrix @ embeddings_matrix[idx_ejemplo]
orden = np.argsort(-sims)
vecinos = [ids_con_texto[i] for i in orden[1:6]]

def resumen_texto(idp, max_chars=220):
    texto = doc_por_persona.get(idp, "")
    return texto.replace(chr(10), " | ")[:max_chars]

print(f"Persona de referencia {persona_ejemplo}:")
print(" ", resumen_texto(persona_ejemplo))
print()
print("Vecinos mas cercanos (similitud coseno, documento semantico completo):")
for v in vecinos:
    print(f"  [{sims[list(ids_con_texto).index(v)]:.3f}] {v}: {resumen_texto(v)}")

## 7. Embedding de TRAYECTORIA PROFESIONAL (capa nueva, separada del embedding general)

Segunda capa de embedding, **independiente** del documento semántico general (secciones 5-6):
un embedding calculado exclusivamente sobre la trayectoria de cargos/unidades de cada persona,
para permitir búsquedas orientadas específicamente a patrones de carrera (estabilidad,
movilidad, permanencia) en vez de a temas/competencias.

Arquitectura:

```
IDPERSONA
    ├── embedding general (secciones 5-6, arriba)   — QUÉ sabe/hace la persona (temas, formación, producción)
    └── embedding de trayectoria (esta sección)      — CÓMO ha sido su carrera (cargos, permanencia, movilidad)
             ↑
             ├── tramos_rol.csv (fuente única de cargo/unidad/fechas, ya calculada en 04_trayectorias.ipynb)
             ├── conocimiento estructurado existente (personas_dashboard.csv: estado de vigencia actual)
             ├── consolidación de tramos por (cargo, unidad, receso corto)
             ├── duración de cargos (media/mediana/máxima)
             ├── cambios de cargo vs. cambios de unidad (nunca confundidos entre sí)
             ├── estabilidad (reglas explícitas y configurables)
             └── movilidad (reglas explícitas y configurables)
```

**No sustituye ni modifica el embedding general**: es un vector adicional por persona, guardado
en un archivo aparte (`embeddings_trayectoria.csv`). El clustering estructural (`06_clustering`)
tampoco se toca — misma separación de responsabilidades que ya rige el resto de este notebook.

**Fuentes reutilizadas (sin recalcular nada que ya exista):**
- `data/trayectorias/tramos_rol.csv` — cargo, unidad, fecha de inicio/fin por tramo (ya
  consolidado por `CATEGORIA_CARGO`, ver `pc.construir_tramos_rol`).
- `data/dashboard/personas_dashboard.csv` — el conocimiento estructurado más completo del
  proyecto (106 columnas, incluye ya `CARGO_ACTUAL`, `UNIDAD_ACTUAL_NOMBRE`,
  `VIGENTE_TRAMO_ESTRUCTURAL`), usado únicamente para narrar el **estado actual** en el texto
  (no se recalculan `N_CARGOS_DISTINTOS`/`N_UNIDADES_DISTINTAS`/`DURACION_MEDIANA_TRAMO_ANIOS`
  ya existentes ahí: esas cuentan sobre el historial crudo sin la consolidación cargo+unidad
  que pide esta tarea, así que las variables de esta sección son deliberadamente más finas y
  se calculan aparte, documentado explícitamente en `_embeddings_comun.py`).

**Por qué se re-consolidan los tramos (y no se usa `tramos_rol.csv` tal cual):**
`tramos_rol.csv` ya fusiona contratos por `CATEGORIA_CARGO` (una categoría de rol amplia, p.ej.
"DOCENTE_TITULAR_CARRERA"), pero **no** por el cargo/unidad textual exacto. Una persona con
contratación por periodo académico (ej. "Profesor Pregrado" renovado cada semestre durante 10
años) aparece en `tramos_rol.csv` con un tramo nuevo por cada semestre, separados por el receso
de vacaciones — lo que infla artificialmente `N_CARGOS_TOTAL` y hace parecer inestable a alguien
con una permanencia real de una década (mismo patrón que ya motivó una corrección equivalente en
`dashboard_react/backend/main.py` para el conteo de cargos del dashboard, 2026-09-16). Esta
sección aplica la misma idea (fusionar por receso corto, `TOLERANCIA_RECESO_DIAS_CARGO = 90
días`) pero exige además coincidencia exacta de **cargo + unidad**, no solo de categoría — así
un cambio real de unidad (aunque el cargo textual sea igual) sigue contando como cambio de
unidad, y un cambio real de cargo (aunque la unidad sea igual) sigue contando como cambio de
cargo, sin confundir ambos conceptos (pedido explícito).

**Umbral configurable:** `MIN_MESES_CARGO_SIGNIFICATIVO = 3` (definido como constante en
`_embeddings_comun.py`, no escrito repetidamente en el código) determina qué cargos cuentan
como "significativos" para los indicadores de permanencia/estabilidad — un cargo corto sigue
apareciendo en la secuencia cronológica del texto, solo no cuenta para esos indicadores.

In [ ]:
tramos_rol = pd.read_csv(DATA_TRAYECTORIAS / "tramos_rol.csv", low_memory=False)
for _c in ["TRAMO_INICIO", "TRAMO_FIN"]:
    tramos_rol[_c] = pd.to_datetime(tramos_rol[_c], format="mixed", errors="coerce")

# Conocimiento estructurado EXISTENTE (no se recalcula nada de aqui): solo se usa
# CARGO_ACTUAL/UNIDAD_ACTUAL_NOMBRE/VIGENTE_TRAMO_ESTRUCTURAL para narrar el estado actual.
personas_dashboard = pd.read_csv(
    ROOT / "data" / "dashboard" / "personas_dashboard.csv",
    usecols=["IDPERSONA", "CARGO_ACTUAL", "UNIDAD_ACTUAL_NOMBRE", "VIGENTE_TRAMO_ESTRUCTURAL"],
)

documentos_trayectoria = ec.construir_documento_trayectoria(tramos_rol, personas_dashboard, poblacion=POBLACION)

print(f"Personas con tramo de rol estructural (candidatas a documento de trayectoria): {len(documentos_trayectoria)} "
      f"de {len(POBLACION)} ({100*len(documentos_trayectoria)/len(POBLACION):.1f}%)")
print("(El resto no tiene ningun tramo estructural - solo tuvo contratos de categorias puntuales/por proyecto, "
      "ver CATEGORIAS_PUNTUALES/DEC-004 - mismo universo que features_trayectoria_persona.csv)")
print()
print("Variables objetivas de trayectoria (describe()):")
documentos_trayectoria[[
    "N_CARGOS_TOTAL", "N_CARGOS_SIGNIFICATIVOS", "N_CAMBIOS_CARGO", "N_CAMBIOS_UNIDAD",
    "DURACION_MEDIA_CARGO_ANIOS", "DURACION_MEDIANA_CARGO_ANIOS", "DURACION_MAX_CARGO_ANIOS",
    "N_UNIDADES_TOTAL", "N_UNIDADES_SIGNIFICATIVAS", "PROPORCION_CARGOS_SIGNIFICATIVOS",
]].describe()

In [ ]:
documentos_trayectoria.to_csv(
    DATA_EMBEDDINGS / "documento_trayectoria_persona.csv", index=False, encoding="utf-8-sig"
)
print(f"Guardado: {DATA_EMBEDDINGS / 'documento_trayectoria_persona.csv'} — {documentos_trayectoria.shape}")

vacios_trayectoria = (documentos_trayectoria["DOCUMENTO_TRAYECTORIA_TEXTO"].str.len() == 0).sum()
print(f"Personas con tramo pero sin texto generable (caso degenerado, no debería ocurrir): {vacios_trayectoria}")
print()
print("--- Ejemplo: persona con más cargos consolidados ---")
ej = documentos_trayectoria.loc[documentos_trayectoria["N_CARGOS_TOTAL"].idxmax()]
print(f"IDPERSONA {ej['IDPERSONA']} (N_CARGOS_TOTAL={ej['N_CARGOS_TOTAL']}, "
      f"N_CAMBIOS_CARGO={ej['N_CAMBIOS_CARGO']}, N_CAMBIOS_UNIDAD={ej['N_CAMBIOS_UNIDAD']})")
print(ej["DOCUMENTO_TRAYECTORIA_TEXTO"])

### 7.0b Tramos consolidados por cargo+unidad (tabla estructurada, para la ficha del dashboard)

Se exporta como tabla aparte el mismo resultado intermedio que ya usó la sección anterior
para armar el texto (`_consolidar_tramos_cargo_unidad`), con detección de cargos paralelos
(dos tramos consolidados de la misma persona que se solapan en fecha con cargo o unidad
distintos). Pensado para que el dashboard muestre "períodos, cargos y unidades" en la ficha
de una persona sin recalcular nada — no es una nueva fuente de datos, es la misma
consolidación ya validada en 7.2, expuesta como CSV.

In [ ]:
tramos_cargo_unidad = ec.construir_tramos_cargo_unidad_persona(tramos_rol, poblacion=POBLACION)
tramos_cargo_unidad.to_csv(
    DATA_EMBEDDINGS / "tramos_cargo_unidad_persona.csv", index=False, encoding="utf-8-sig"
)
print(f"Guardado: {DATA_EMBEDDINGS / 'tramos_cargo_unidad_persona.csv'} — {tramos_cargo_unidad.shape}")
print(f"Personas con al menos un cargo paralelo detectado: {tramos_cargo_unidad[tramos_cargo_unidad['ES_PARALELO']]['IDPERSONA'].nunique()}")
tramos_cargo_unidad.head(10)

### 7.1 Cálculo del embedding de trayectoria

Se reutiliza el **mismo modelo** ya cargado en la sección 6 (`model`, `intfloat/multilingual-e5-base`)
— no se instancia un segundo modelo. Mismo prefijo `"passage: "` para consistencia con el
embedding general y con el prefijo `"query: "` que deberá usarse en cualquier búsqueda futura
sobre este índice.

In [ ]:
con_texto_trayectoria = documentos_trayectoria[documentos_trayectoria["DOCUMENTO_TRAYECTORIA_TEXTO"].str.len() > 0].copy()
sin_embedding_trayectoria = sorted(POBLACION - set(con_texto_trayectoria["IDPERSONA"]))

textos_passage_trayectoria = ("passage: " + con_texto_trayectoria["DOCUMENTO_TRAYECTORIA_TEXTO"]).tolist()
print(f"Codificando {len(textos_passage_trayectoria)} documentos de trayectoria...")

embeddings_matrix_trayectoria = model.encode(
    textos_passage_trayectoria, batch_size=32, show_progress_bar=True, normalize_embeddings=True,
)
ids_con_texto_trayectoria = con_texto_trayectoria["IDPERSONA"].to_numpy()

print("Embeddings de trayectoria calculados:", embeddings_matrix_trayectoria.shape)
print(f"Personas SIN embedding de trayectoria (sin tramo de rol o documento vacío): {len(sin_embedding_trayectoria)}")

In [ ]:
embeddings_trayectoria = pd.DataFrame(
    embeddings_matrix_trayectoria, columns=[f"E_{i:03d}" for i in range(embeddings_matrix_trayectoria.shape[1])],
)
embeddings_trayectoria.insert(0, "IDPERSONA", ids_con_texto_trayectoria)

embeddings_trayectoria.to_csv(DATA_EMBEDDINGS / "embeddings_trayectoria.csv", index=False)

metadata_embeddings_trayectoria = pd.DataFrame([{
    "MODELO": MODEL_NAME,
    "DIMENSIONES": embeddings_matrix_trayectoria.shape[1],
    "METODO": "un embedding por persona, calculado sobre el documento de TRAYECTORIA PROFESIONAL "
              "(cargo/unidad/permanencia/estabilidad/movilidad) - independiente del embedding "
              "general (documento_semantico_persona.csv); no se concatenan ni se fusionan",
    "PREFIJO_DOCUMENTO": "passage: ",
    "PREFIJO_CONSULTA_BUSQUEDA": "query: ",
    "MIN_MESES_CARGO_SIGNIFICATIVO": ec.MIN_MESES_CARGO_SIGNIFICATIVO,
    "TOLERANCIA_RECESO_DIAS_CARGO": ec.TOLERANCIA_RECESO_DIAS_CARGO,
    "N_PERSONAS_CON_EMBEDDING": len(ids_con_texto_trayectoria),
    "N_PERSONAS_SIN_EMBEDDING": len(sin_embedding_trayectoria),
}])
metadata_embeddings_trayectoria.to_csv(DATA_EMBEDDINGS / "embeddings_trayectoria_metadata.csv", index=False)

print(f"Guardado: {DATA_EMBEDDINGS / 'embeddings_trayectoria.csv'} — {embeddings_trayectoria.shape}")
print(f"Guardado: {DATA_EMBEDDINGS / 'embeddings_trayectoria_metadata.csv'}")
metadata_embeddings_trayectoria.T

### 7.2 Validación manual: casos contrastantes de trayectoria

Antes de aceptar el embedding de trayectoria, se revisan manualmente 5 perfiles con patrones
claramente distintos (identificados por sus variables objetivas, no elegidos al azar), para
confirmar que el texto generado refleja fielmente lo que dicen los datos y que los cargos cortos
no inflan artificialmente `N_CARGOS_SIGNIFICATIVOS` ni los indicadores de estabilidad:

1. Pocos cargos y permanencias largas (alta estabilidad esperada).
2. Muchos cargos de corta duración (baja estabilidad esperada).
3. Varios cambios de unidad manteniendo un cargo similar (movilidad de unidad, no de cargo).
4. Cambios frecuentes tanto de cargo como de unidad (alta movilidad en ambas dimensiones).
5. Historial con varios períodos menores a `MIN_MESES_CARGO_SIGNIFICATIVO` (3 meses).

In [ ]:
# Seleccion automatica de un caso representativo por patron (a partir de las variables
# objetivas ya calculadas, no elegidos a mano) - reproducible sin depender de IDPERSONA fijos.
d = documentos_trayectoria

caso_estable = d[(d["N_CARGOS_TOTAL"] <= 2) & (d["DURACION_MEDIANA_CARGO_ANIOS"] >= 10)] \
    .sort_values("DURACION_MEDIANA_CARGO_ANIOS", ascending=False)
caso_inestable = d[(d["N_CARGOS_TOTAL"] >= 5) & (d["DURACION_MEDIANA_CARGO_ANIOS"] <= 0.5)] \
    .sort_values("N_CARGOS_TOTAL", ascending=False)
caso_movilidad_unidad = d[(d["N_CAMBIOS_UNIDAD"] >= 3) & (d["N_CAMBIOS_CARGO"] <= 1)] \
    .sort_values("N_CAMBIOS_UNIDAD", ascending=False)
caso_movilidad_ambas = d[(d["N_CAMBIOS_CARGO"] >= 5) & (d["N_CAMBIOS_UNIDAD"] >= 3)] \
    .sort_values("N_CAMBIOS_CARGO", ascending=False)
caso_periodos_cortos = d[(d["N_CARGOS_TOTAL"] >= 5) & (d["PROPORCION_CARGOS_SIGNIFICATIVOS"] <= 0.4)] \
    .sort_values("N_CARGOS_TOTAL", ascending=False)

casos_validacion = {
    "1. Pocos cargos, permanencia larga (alta estabilidad)": caso_estable,
    "2. Muchos cargos cortos (baja estabilidad)": caso_inestable,
    "3. Cambios de unidad manteniendo cargo similar": caso_movilidad_unidad,
    "4. Cambios frecuentes de cargo Y unidad": caso_movilidad_ambas,
    "5. Varios periodos menores a MIN_MESES_CARGO_SIGNIFICATIVO": caso_periodos_cortos,
}

for nombre, candidatos in casos_validacion.items():
    print("=" * 100)
    print(nombre)
    print("=" * 100)
    if candidatos.empty:
        print("(sin candidatos que cumplan el patron en la poblacion actual)")
        print()
        continue
    r = candidatos.iloc[0]
    print(f"IDPERSONA {int(r['IDPERSONA'])}  |  N_CARGOS_TOTAL={r['N_CARGOS_TOTAL']}  "
          f"N_CARGOS_SIGNIFICATIVOS={r['N_CARGOS_SIGNIFICATIVOS']}  "
          f"N_CAMBIOS_CARGO={r['N_CAMBIOS_CARGO']}  N_CAMBIOS_UNIDAD={r['N_CAMBIOS_UNIDAD']}  "
          f"PROPORCION_SIGNIFICATIVOS={r['PROPORCION_CARGOS_SIGNIFICATIVOS']:.2f}")
    print()
    print(r["DOCUMENTO_TRAYECTORIA_TEXTO"])
    print()

### 7.3 Validación cualitativa: vecinos más cercanos en el espacio de trayectoria

Misma verificación de sentido común que en la sección 6 para el embedding general: para una
persona con trayectoria distintiva (varios cargos, movilidad clara), ¿sus vecinos más cercanos
por similitud coseno en el espacio de TRAYECTORIA tienen un patrón de carrera parecido (no
necesariamente el mismo cargo, sino un patrón similar de estabilidad/movilidad)?

In [ ]:
doc_trayectoria_por_persona = documentos_trayectoria.set_index("IDPERSONA")["DOCUMENTO_TRAYECTORIA_TEXTO"]

# Referencia: el caso "movilidad en ambas dimensiones" (seccion 7.2), tipicamente el mas
# distintivo de los cinco patrones.
if not caso_movilidad_ambas.empty:
    persona_ref_trayectoria = int(caso_movilidad_ambas.iloc[0]["IDPERSONA"])
    idx_ref = list(ids_con_texto_trayectoria).index(persona_ref_trayectoria)
    sims_trayectoria = embeddings_matrix_trayectoria @ embeddings_matrix_trayectoria[idx_ref]
    orden_trayectoria = np.argsort(-sims_trayectoria)
    vecinos_trayectoria = [ids_con_texto_trayectoria[i] for i in orden_trayectoria[1:6]]

    def resumen_trayectoria(idp, max_chars=260):
        texto = doc_trayectoria_por_persona.get(idp, "")
        return texto.replace(chr(10), " | ")[:max_chars]

    print(f"Persona de referencia {persona_ref_trayectoria} (alta movilidad de cargo y unidad):")
    print(" ", resumen_trayectoria(persona_ref_trayectoria))
    print()
    print("Vecinos mas cercanos en el espacio de TRAYECTORIA (similitud coseno):")
    for v in vecinos_trayectoria:
        print(f"  [{sims_trayectoria[list(ids_con_texto_trayectoria).index(v)]:.3f}] {v}: {resumen_trayectoria(v)}")
else:
    print("Sin candidato de referencia para esta verificacion en la poblacion actual.")

## 8. ¿Los embeddings identifican una estructura distinta a la de `06_clustering`?

Se agrupa el espacio de embeddings por sí solo (K-Means exploratorio, K=5) y se compara,
vía ARI, contra la jerarquía estructural actual (`data/clustering/clusters_personas.csv`,
columna `CLUSTER` — 13 categorías de cargo real, DEC-009, ya no un K-Means; ARI no requiere
que ambas particiones tengan el mismo número de grupos). La comparación se limita a las
personas que tienen documento semántico con al menos una sección (no todas las 3195 tienen
embedding).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

clusters_estructurales = pd.read_csv(ROOT / "data" / "clustering" / "clusters_personas.csv")

X_emb = embeddings_matrix  # ya normalizado por fila

filas_emb_k = []
for k in range(3, 9):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_emb)
    filas_emb_k.append({"K": k, "SILHOUETTE": silhouette_score(X_emb, labels)})
pd.DataFrame(filas_emb_k)

Se usa **K=5** para el agrupamiento exploratorio del espacio de embeddings — un valor
razonable para inspeccionar la estructura semántica por sí sola, sin intentar igualar el
número de categorías de la jerarquía estructural actual (13 categorías de cargo real,
DEC-009): ARI compara la concordancia entre dos particiones sin importar que tengan
cardinalidades distintas.

In [ ]:
labels_emb = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE).fit_predict(X_emb)

emb_clusters_df = pd.DataFrame({"IDPERSONA": ids_con_texto, "CLUSTER_EMBEDDING": labels_emb})
comparacion = emb_clusters_df.merge(clusters_estructurales, on="IDPERSONA", how="inner")
comparacion = comparacion.rename(columns={"CLUSTER": "CLUSTER_ESTRUCTURAL"})

ari = adjusted_rand_score(comparacion["CLUSTER_ESTRUCTURAL"], comparacion["CLUSTER_EMBEDDING"])
print(f"Personas comparadas: {len(comparacion)}")
print(f"ARI (estructural [13 categorías, DEC-009] vs. embeddings [K=5 exploratorio]): {ari:.3f}")
print()
tabla_cruzada = pd.crosstab(comparacion["CLUSTER_ESTRUCTURAL"], comparacion["CLUSTER_EMBEDDING"])
tabla_cruzada


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

pca_emb = PCA(n_components=2, random_state=RANDOM_STATE)
X_emb_pca = pca_emb.fit_transform(X_emb)

paleta5 = sns.color_palette("Set2", 5)
merged_pca = pd.DataFrame(X_emb_pca, columns=["PC1", "PC2"])
merged_pca["IDPERSONA"] = ids_con_texto
merged_pca = merged_pca.merge(comparacion, on="IDPERSONA", how="left")

for c in range(5):
    m = merged_pca["CLUSTER_EMBEDDING"] == c
    axes[0].scatter(merged_pca.loc[m, "PC1"], merged_pca.loc[m, "PC2"], s=10, alpha=0.6, color=paleta5[c], label=f"C{c}")
axes[0].set_title("Espacio de embeddings (PCA 2D)\ncoloreado por cluster DE EMBEDDINGS")
axes[0].legend(fontsize=8, markerscale=2)

for c in range(5):
    m = merged_pca["CLUSTER_ESTRUCTURAL"] == c
    axes[1].scatter(merged_pca.loc[m, "PC1"], merged_pca.loc[m, "PC2"], s=10, alpha=0.6, color=paleta5[c], label=f"C{c}")
axes[1].set_title("Espacio de embeddings (PCA 2D)\ncoloreado por cluster ESTRUCTURAL (05)")
axes[1].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.show()


**Lectura:** un ARI cercano a 0 indicaría que los embeddings capturan una estructura
prácticamente independiente de la estructural (alta complementariedad); un ARI cercano a 1
indicaría que agrupan a las personas de forma casi idéntica (redundancia, poco aporte adicional).
El valor obtenido (ver celda anterior) y el gráfico de la derecha — que muestra si los clusters
estructurales forman regiones reconocibles o aparecen mezclados en el espacio semántico — se toman
en conjunto para la conclusión de la sección 9.


## 9. ¿Aportan los embeddings algo que las features estructuradas no capturan?

Además del ARI global, conviene revisar si el espacio semántico distingue **dentro** de un mismo
cluster estructural — por ejemplo, dos docentes de "alta carga docente" (mismo Cluster 3 en `05`)
pueden enseñar/investigar en dominios completamente distintos (ingeniería vs. ciencias sociales), y
esa diferencia de dominio es justamente lo que las 100 columnas de `X_modelado` no capturan (son
conteos de actividad, no de contenido/tema).


In [ ]:
# Dispersión semántica dentro de cada cluster estructural: similitud coseno promedio
# entre pares de personas del mismo cluster vs. pares de clusters distintos.
from itertools import combinations
rng = np.random.default_rng(42)

def similitud_promedio_pares(indices, n_pares=300):
    if len(indices) < 2:
        return np.nan
    pares = rng.choice(len(indices), size=(min(n_pares, len(indices) * (len(indices) - 1) // 2), 2))
    pares = pares[pares[:, 0] != pares[:, 1]]
    sims = [float(X_emb[indices[i]] @ X_emb[indices[j]]) for i, j in pares]
    return np.mean(sims)

idx_por_cluster_estructural = {
    c: [list(ids_con_texto).index(i) for i in comparacion.loc[comparacion["CLUSTER_ESTRUCTURAL"] == c, "IDPERSONA"]]
    for c in sorted(comparacion["CLUSTER_ESTRUCTURAL"].unique())
}

filas_dispersión = []
for c, idxs in idx_por_cluster_estructural.items():
    filas_dispersión.append({
        "CLUSTER_ESTRUCTURAL": c,
        "N_PERSONAS": len(idxs),
        "SIMILITUD_SEMANTICA_INTRA_CLUSTER": similitud_promedio_pares(idxs),
    })

todos_los_idx = list(range(len(ids_con_texto)))
sim_global = similitud_promedio_pares(todos_los_idx, n_pares=500)

dispersión_df = pd.DataFrame(filas_dispersión)
print(f"Similitud semántica promedio entre dos personas cualesquiera (referencia global): {sim_global:.3f}")
dispersión_df


**Interpretación:** si la similitud semántica promedio **dentro** de cada cluster estructural
fuera muy superior a la similitud global de referencia, significaría que el clustering estructural
(basado en volumen/tipo de actividad) ya agrupa implícitamente a personas con temas afines — los
embeddings aportarían poco. Si en cambio la similitud intra-cluster es parecida a la global, cada
cluster estructural mezcla temas muy distintos entre sí, y una representación semántica sí aportaría
una dimensión de información **adicional** (a qué se dedica temáticamente cada persona) que
`X_modelado` no captura.

**Resultado obtenido:** ARI = **0.116** entre la partición estructural y la de embeddings (ambas
K=5) — muy lejos de 1, señal de particiones prácticamente independientes. La similitud semántica
intra-cluster (referencia global: 0.693) es:

- Cluster 1 (perfil administrativo): 0.786 — el más homogéneo temáticamente, esperable dado que el
  vocabulario administrativo/laboral es más acotado que el académico.
- Clusters 0, 2 y 3 (perfiles docentes): 0.72-0.75 — apenas por encima del global, es decir, dentro
  de cada uno de estos clusters conviven temas/disciplinas bastante distintos.
- Cluster 4 (ingreso reciente): 0.578 — el más heterogéneo temáticamente, incluso por debajo del
  promedio global: el criterio que define este cluster (poca antigüedad) no tiene ninguna relación
  con el área de conocimiento de la persona.

**Conclusión de esta exploración (06):**

- Los embeddings **sí capturan información distinta** a la estructural: el ARI de 0.116 confirma
  particiones mayormente independientes, y salvo el caso parcial del cluster administrativo, la
  similitud intra-cluster estructural no supera de forma relevante a la similitud global — es decir,
  los clusters estructurales (definidos por volumen/tipo de actividad: docencia, investigación,
  administración, antigüedad) en general NO predicen bien de qué tema/disciplina trata el trabajo de
  cada persona. Ambas representaciones son **complementarias**, no redundantes: la estructural
  describe *cuánto y qué tipo* de actividad tiene una persona; la semántica describe *sobre qué*
  trata esa actividad.
- **No se concatena una matriz combinada (100 + 384 dimensiones) en este notebook.** Aunque hay
  evidencia de complementariedad, concatenar directamente estructurado + embeddings sin un análisis
  adicional (p. ej. reducir la dimensionalidad de los embeddings para no dominar la distancia
  euclídea frente a las 100 columnas, decidir una ponderación relativa, y re-evaluar
  estabilidad/interpretabilidad del clustering resultante) no está metodológicamente justificado
  todavía — así lo pide la consigna del proyecto. Queda documentado como una línea de trabajo futura
  concreta, no como algo ya resuelto.
- **Uso recomendado en el corto plazo:** mantener ambas representaciones **separadas**. Los perfiles
  estructurales de `06_clustering` siguen siendo la base para el dashboard (`08`); los embeddings de
  este notebook pueden usarse de forma independiente para funciones complementarias como "buscar
  personas con experiencia temática similar a X" (búsqueda semántica) dentro de cada perfil
  estructural, sin fusionar ambas matrices.

## 10. Resumen

**Archivos generados en `data/embeddings/`:**

| Archivo | Contenido |
|---|---|
| `documento_semantico_persona.csv` | `IDPERSONA, DOCUMENTO_TEXTO, SECCIONES_INCLUIDAS, SECCIONES_RECORTADAS, N_SECCIONES, N_CARACTERES, N_PALABRAS` — el documento GENERAL que se embebe (DEC-014) |
| `corpus_texto_detalle.csv` | `IDPERSONA, FUENTE, TEXTO` — un registro por fragmento, usado por el dashboard como evidencia, no como insumo del embedding |
| `cobertura_texto_personas.csv` | Cobertura de texto por persona (conteos), del corpus por fragmento |
| `embeddings_personas.csv` | `IDPERSONA` + 768 columnas `E_000..E_767` (embedding GENERAL, normalizado, `intfloat/multilingual-e5-base`) |
| `embeddings_metadata.csv` | Modelo, dimensiones, método, cobertura del embedding general |
| `documento_trayectoria_persona.csv` | `IDPERSONA, DOCUMENTO_TRAYECTORIA_TEXTO` + variables objetivas (`N_CARGOS_TOTAL`, `N_CARGOS_SIGNIFICATIVOS`, `N_CAMBIOS_CARGO`, `N_CAMBIOS_UNIDAD`, duraciones, unidades) — el documento de TRAYECTORIA (sección 7, capa nueva) |
| `embeddings_trayectoria.csv` | `IDPERSONA` + 768 columnas `E_000..E_767` (embedding de TRAYECTORIA, independiente del general) |
| `embeddings_trayectoria_metadata.csv` | Modelo, dimensiones, método, umbrales configurados (`MIN_MESES_CARGO_SIGNIFICATIVO`, `TOLERANCIA_RECESO_DIAS_CARGO`), cobertura |

**Decisiones tomadas (DEC-014, ver `context/DECISION_LOG.md`) — embedding GENERAL:**

1. Se reemplazó la agregación de fragmentos (promedio balanceado por fuente) por un
   **documento semántico único por persona**, construido por secciones con información
   real, conservando la relación temporal entre trayectoria dentro/fuera de ESPOL y
   distinguiendo cargo contractual de función adicional/subrogación.
2. Se cambió el modelo de `paraphrase-multilingual-MiniLM-L12-v2` (128 tokens, tuneado
   para parafraseo/STS) a **`intfloat/multilingual-e5-base`** (512 tokens, tuneado para
   recuperación/búsqueda semántica) — decisión técnica: el documento narrativo completo
   no cabía en 128 tokens. Ambos son locales, sin API key (se mantiene DEC-002).
3. Se aplicó un presupuesto de longitud (~380 palabras) que recorta secciones completas de
   menor prioridad (nunca trayectoria ni formación) para las personas con historiales más
   extensos, en vez de dejar que el tokenizador trunque a mitad de frase de forma opaca.
4. Se comparó la estructura de los embeddings contra la jerarquía estructural actual y se
   concluyó que son **complementarios**: no se fusionan en una sola matriz (instrucción
   explícita del usuario de no mezclar clustering con embeddings).

**Decisiones tomadas — embedding de TRAYECTORIA (sección 7, capa nueva):**

1. Se reutilizó `tramos_rol.csv` (única fuente de cargo/unidad/fechas, ya calculada en
   `04_trayectorias.ipynb`) y `personas_dashboard.csv` (conocimiento estructurado existente,
   usado solo para narrar el estado de vigencia actual) — no se creó ninguna fuente ni tabla
   estructurada nueva.
2. Se detectó que `tramos_rol.csv` fusiona por `CATEGORIA_CARGO` pero no por cargo/unidad
   textual exacto, así que una persona con contratación por periodo académico aparecía con
   decenas de tramos fragmentados del mismo cargo real. Se implementó una consolidación
   adicional (`_consolidar_tramos_cargo_unidad`, rastreo en paralelo por clave (cargo,
   unidad), mismo patrón que `pc.construir_tramos_rol`) con una tolerancia de receso corto
   (`TOLERANCIA_RECESO_DIAS_CARGO = 90` días, misma magnitud que la corrección equivalente
   ya aplicada en `dashboard_react/backend/main.py`).
3. Se calcularon variables objetivas nuevas y más finas que las ya existentes en el
   conocimiento estructurado (`N_CARGOS_TOTAL`, `N_CARGOS_SIGNIFICATIVOS`, `N_CAMBIOS_CARGO`,
   `N_CAMBIOS_UNIDAD` — nunca confundidos entre sí —, duración media/mediana/máxima, unidades
   totales y significativas), con el umbral `MIN_MESES_CARGO_SIGNIFICATIVO = 3` configurable
   como constante en `_embeddings_comun.py` (no repetido en el código).
4. Se definieron reglas explícitas y configurables (no etiquetas arbitrarias) para describir
   estabilidad y movilidad, calibradas contra la distribución real de la población antes de
   fijar los umbrales (ver sección 7, celda de calibración): duración mediana ≥ 2.0 años y
   proporción de cargos significativos ≥ 0.7 para "alta estabilidad"; ≥ 3 cambios de cargo o
   ≥ 2 cambios de unidad para "alta movilidad" en cada dimensión, evaluadas de forma
   independiente para no confundir movilidad funcional (cargo) con movilidad organizacional
   (unidad).
5. Se generaron insights automáticos (permanencia prolongada, múltiples cambios de cargo/
   unidad, concentración/dispersión entre unidades, predominio de períodos cortos/largos) a
   partir de las mismas reglas explícitas, sin descripciones subjetivas.
6. Se calculó un **segundo embedding independiente** (mismo modelo, mismo prefijo `passage:`)
   sobre este documento de trayectoria — no reemplaza ni se concatena con el embedding
   general, y el clustering estructural (`06_clustering`) no se modifica.
7. Validado manualmente contra 5 casos contrastantes (sección 7.2: pocos cargos/permanencia
   larga, muchos cargos cortos, movilidad de unidad sin movilidad de cargo, movilidad alta en
   ambas dimensiones, y periodos menores al umbral de significatividad) — el texto generado
   refleja fielmente el patrón de cada caso, y la consolidación por cargo+unidad+receso evita
   que los cargos cortos por renovación semestral inflen artificialmente
   `N_CARGOS_SIGNIFICATIVOS` (caso de referencia: una persona con 29 tramos crudos en
   `tramos_rol.csv` se consolida a 7 cargos reales, con la duración mediana subiendo de 1.27 a
   3.45 años tras la consolidación correcta).

**Limitaciones:**

- Personas sin ninguna sección con información (embedding general) o sin ningún tramo de rol
  estructural (embedding de trayectoria — solo tuvieron contratos de categorías puntuales,
  ver `CATEGORIAS_PUNTUALES`/DEC-004) no tienen documento ni embedding en la capa
  correspondiente — deben tratarse explícitamente en cualquier uso posterior.
- El presupuesto de longitud (embedding general) recorta secciones completas de menor
  prioridad (nunca trayectoria/formación) para el ~20% de la población con historiales más
  extensos (ver sección 5); el recorte aplica tanto al texto guardado en
  `documento_semantico_persona.csv` como al embedding — no se conserva una versión "completa"
  sin recortar aparte. El detalle completo de esas secciones sigue disponible en las fuentes
  originales (`data/processed/*.csv`) y en `corpus_texto_detalle.csv`.
- No se evaluaron alternativas de modelo con contexto aún mayor (p. ej. `BAAI/bge-m3`,
  8192 tokens) ni pooling ponderado por recencia; son posibles mejoras futuras.
- El embedding de trayectoria no se comparó cuantitativamente (ARI/similitud intra-cluster)
  contra el clustering estructural ni contra el embedding general en esta pasada — el pedido
  explícito de esta tarea fue dejar la representación funcionando con variables objetivas y
  validación manual, no un ranking híbrido ni un nuevo sistema de interpretación de consultas
  (fuera de alcance, ver restricciones de la tarea).